# RF K-Fold x RF-Top Features K-Fold

## 1. Importing the Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap 

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict

## 2. Data Loading and Exploratory Data Analysis

In [ ]:
# load dataset
file_path = '/kaggle/input/datasets/ikynahidwin/depression-student-dataset/Depression Student Dataset.csv'
df = pd.read_csv(file_path)

In [ ]:
# display the first 5 lines
display(df.head())

In [ ]:
# check the data type and missing values
print("\n--- Info Dataset ---")
print(df.info())
print("\n--- Missing Values ---")
print(df.isnull().sum())

In [ ]:
# Plot distribusi Class/Target (Depression)
plt.figure(figsize=(5, 4))
sns.countplot(data=df, x='Depression', hue='Depression', palette='Set2', legend=False)
plt.title('Distribusi Target (Depression)')
plt.show()

In [ ]:
# Separating column types for EDA
num_cols = ['Age', 'Study Hours', 'Academic Pressure', 'Study Satisfaction', 'Financial Stress']
cat_cols = [
    'Gender', 'Sleep Duration', 'Dietary Habits', 
    'Have you ever had suicidal thoughts ?', 'Family History of Mental Illness'
]

In [ ]:
# EDA numerical col (Distribusi X terhadap targer)
print("\n--- Distribusi Fitur Numerik ---")
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.histplot(data=df, x=col, kde=True, hue='Depression', ax=axes[i], palette='Set2', multiple="stack")
    axes[i].set_title(f'Distribusi {col}')

fig.delaxes(axes[-1]) 
plt.tight_layout()
plt.show()

In [ ]:
# EDA Fitur Kategorikal (Distribusi X terhadap Target)
print("\n--- Distribusi Fitur Kategorikal vs Target ---")
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    sns.countplot(data=df, x=col, hue='Depression', ax=axes[i], palette='Set2')
    axes[i].set_title(f'{col} vs Depression')
    axes[i].tick_params(axis='x', rotation=15) 

fig.delaxes(axes[-1]) # Menghapus subplot terakhir yang kosong
plt.tight_layout()
plt.show()

## 3. Data Pre-Processing

In [ ]:
df_processed = df.copy()

In [ ]:
# search categorical column
categorical_cols = df_processed.select_dtypes(include=['object']).columns.tolist()

# label endcoding
le = LabelEncoder()
for col in categorical_cols:
    df_processed[col] = le.fit_transform(df_processed[col])

In [ ]:
# separating features (X) dan target (Y)
X = df_processed.drop('Depression', axis=1)
y = df_processed['Depression']

## 4. Splitting the Dataset into Training Set and Test Set

In [ ]:
# using stratify=y to balance train and test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=40, stratify=y)

In [ ]:
# print the dimension X_train and X_test
print("Dimensi X_train:", X_train.shape)
print("Dimensi X_test:", X_test.shape)

## 5. Train and Evaluate the Model using Random Forest as Baseline Using All Feature

In [ ]:
# initialize K-Fold (5)
# using stratified to ensure the ratio of depressed vs. non-depressed is balanced in each fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
# initialize the model
rf_baseline = RandomForestClassifier(n_estimators=100, random_state=42)

In [ ]:
# determine the metrics you want to measure
scoring = ['accuracy', 'precision', 'recall', 'f1']

In [ ]:
# do K-Fold Cross Validation
cv_results_base = cross_validate(rf_baseline, X, y, cv=cv, scoring=scoring)

In [ ]:
# calculate the average from 5 results
acc_cv_base = cv_results_base['test_accuracy'].mean()
prec_cv_base = cv_results_base['test_precision'].mean()
rec_cv_base = cv_results_base['test_recall'].mean()
f1_cv_base = cv_results_base['test_f1'].mean()

print("--- RATA-RATA METRIK RF BASELINE (5-FOLD) ---")
print(f"Accuracy  : {acc_cv_base:.4f}")
print(f"Precision : {prec_cv_base:.4f}")
print(f"Recall    : {rec_cv_base:.4f}")
print(f"F1-Score  : {f1_cv_base:.4f}\n")

In [ ]:
# creating a combined Confusion Matrix of all K-Fold
# cross_val_predict predict the data when it is the data's turn to be "Test Data"
pred_cv_base_all = cross_val_predict(rf_baseline, X, y, cv=cv)
cm_base = confusion_matrix(y, pred_cv_base_all)

plt.figure(figsize=(6, 4))
sns.heatmap(cm_base, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Tidak Depresi', 'Depresi'], 
            yticklabels=['Tidak Depresi', 'Depresi'])
plt.title('Confusion Matrix Keseluruhan - RF Baseline (5-Fold)')
plt.ylabel('Nilai Aktual (True)')
plt.xlabel('Nilai Prediksi (Predicted)')
plt.show()

## 6. Feature Importance using SHAP

In [ ]:
# training model for all data to analyzed by SHAP
rf_full = RandomForestClassifier(n_estimators=100, random_state=42)
rf_full.fit(X, y)

In [ ]:
# SHAP explainer initialization
explainer = shap.TreeExplainer(rf_full)
shap_values = explainer.shap_values(X)

In [ ]:
# handle the difference between SHAP version
if isinstance(shap_values, list):
    shap_values_target = shap_values[1]
elif len(shap_values.shape) == 3:
    shap_values_target = shap_values[:, :, 1]
else:
    shap_values_target = shap_values

In [ ]:
# SHAP plot summary 
plt.figure(figsize=(8, 5))
plt.title("SHAP Summary Plot")
shap.summary_plot(shap_values_target, X)

In [ ]:
# count SHAP absolute average value to find top features
mean_shap_values = np.abs(shap_values_target).mean(axis=0)

In [ ]:
# making a feature ranking DataFrame
shap_importance_df = pd.DataFrame({
    'Feature': X.columns,
    'SHAP_Importance': mean_shap_values
}).sort_values(by='SHAP_Importance', ascending=False).reset_index(drop=True)

display(shap_importance_df.head(10))

In [ ]:
# taking top-5 best features
top_5_features = shap_importance_df['Feature'].head(5).tolist()
print("\nTop 5 Fitur Paling Penting menurut SHAP:")
for i, f in enumerate(top_5_features, 1):
    print(f"{i}. {f}")

## 7. Train and Evaluate the Model using Random Forest as Baseline Using Top Features

In [ ]:
# filter X_train and X_test for using top 5 features
X_shap = X[top_5_features]

In [ ]:
# running K-Fold Cross Validation for filtered data
cv_results_shap = cross_validate(rf_baseline, X_shap, y, cv=cv, scoring=scoring)

In [ ]:
# count the average  value of 5 test result
acc_cv_shap = cv_results_shap['test_accuracy'].mean()
prec_cv_shap = cv_results_shap['test_precision'].mean()
rec_cv_shap = cv_results_shap['test_recall'].mean()
f1_cv_shap = cv_results_shap['test_f1'].mean()

print("--- RATA-RATA METRIK RF (TOP 5 SHAP) ---")
print(f"Accuracy  : {acc_cv_shap:.4f}")
print(f"Precision : {prec_cv_shap:.4f}")
print(f"Recall    : {rec_cv_shap:.4f}")
print(f"F1-Score  : {f1_cv_shap:.4f}\n")

In [ ]:
# Confusion Matrix combined top 5 SHAP
pred_cv_shap_all = cross_val_predict(rf_baseline, X_shap, y, cv=cv)
cm_shap = confusion_matrix(y, pred_cv_shap_all)

plt.figure(figsize=(6, 4))
sns.heatmap(cm_shap, annot=True, fmt='d', cmap='Oranges', 
            xticklabels=['Tidak Depresi', 'Depresi'], 
            yticklabels=['Tidak Depresi', 'Depresi'])
plt.title('Confusion Matrix - RF (Top 5 SHAP) (5-Fold)')
plt.ylabel('Nilai Aktual (True)')
plt.xlabel('Nilai Prediksi (Predicted)')
plt.show()

## 8. Comparison of Results

In [ ]:
hasil_cv_df = pd.DataFrame({
    'Model': [
        'Random Forest (Baseline K-Fold)', 
        'Random Forest (Top 5 SHAP K-Fold)'
    ],
    'Mean Accuracy': [acc_cv_base, acc_cv_shap],
    'Mean Precision': [prec_cv_base, prec_cv_shap],
    'Mean Recall': [rec_cv_base, rec_cv_shap],
    'Mean F1-Score': [f1_cv_base, f1_cv_shap]
})

print("\nKESIMPULAN PERBANDINGAN MODEL (RATA-RATA K-FOLD):")
display(hasil_cv_df)